# ** THIS NOTEBOOK IS A GARDEN OF EXPLORATION **
                      *** DRAFT of the 01_MARKET_ANALYSIS_AND_SILVENCY_SCOPING ***

## 01 - EDA OF ValeurFonciere-2025
In order to share in the main jupyter notebook the code the most relevant for the understanding and keeping track the topic

In [5]:
# 🇬🇧 Expert Audit: Loading raw DVF data for inspection
# 🇫🇷 Audit Expert : Chargement des données DVF brutes pour inspection

import pandas as pd

# 1. Utilisation de nrows=100 pour une lecture instantanée
# 2. low_memory=False pour éviter les erreurs de type de colonnes
# 3. sep=',' ou '|' (vérifie ton fichier, souvent c'est la virgule ou la barre verticale)

df_audit = pd.read_csv(
    '../data/raw/ValeursFoncieres-2025-S1.txt',
    # nrows=100, 
    sep='|', 
    low_memory=False
)

# Affichage des colonnes pour voir "l'inventaire"
print(df_audit.columns.tolist())

['Identifiant de document', 'Reference document', '1 Articles CGI', '2 Articles CGI', '3 Articles CGI', '4 Articles CGI', '5 Articles CGI', 'No disposition', 'Date mutation', 'Nature mutation', 'Valeur fonciere', 'No voie', 'B/T/Q', 'Type de voie', 'Code voie', 'Voie', 'Code postal', 'Commune', 'Code departement', 'Code commune', 'Prefixe de section', 'Section', 'No plan', 'No Volume', '1er lot', 'Surface Carrez du 1er lot', '2eme lot', 'Surface Carrez du 2eme lot', '3eme lot', 'Surface Carrez du 3eme lot', '4eme lot', 'Surface Carrez du 4eme lot', '5eme lot', 'Surface Carrez du 5eme lot', 'Nombre de lots', 'Code type local', 'Type local', 'Identifiant local', 'Surface reelle bati', 'Nombre pieces principales', 'Nature culture', 'Nature culture speciale', 'Surface terrain']


In [6]:
df_audit.head()

,Identifiant de document,Reference document,1 Articles CGI,2 Articles CGI,3 Articles CGI,4 Articles CGI,5 Articles CGI,No disposition,Date mutation,Nature mutation,...,Surface Carrez du 5eme lot,Nombre de lots,Code type local,Type local,Identifiant local,Surface reelle bati,Nombre pieces principales,Nature culture,Nature culture speciale,Surface terrain
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,07/01/2025,Vente,...,NaN,0,NaN,NaN,NaN,NaN,NaN,J,NaN,78.0
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,07/01/2025,Vente,...,NaN,0,1.0,Maison,NaN,111.0,5.0,S,NaN,133.0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,07/01/2025,Vente,...,NaN,0,3.0,Dépendance,NaN,0.0,0.0,S,NaN,133.0
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,06/01/2025,Vente,...,NaN,0,NaN,NaN,NaN,NaN,NaN,S,NaN,46.0
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,06/01/2025,Vente,...,NaN,0,NaN,NaN,NaN,NaN,NaN,J,NaN,17.0


In [7]:
def check_data_quality(df_audit):

    print(f"Format du DataFrame (rows, cols): {df_audit.shape}\n")
    
    # Création d'un tableau récapitulatif
    stats = pd.DataFrame({
        'Type': df_audit.dtypes,
        'Manquants': df_audit.isna().sum(),
        '% Manquants': (df_audit.isna().sum() / len(df_audit) * 100).round(2),
        'Uniques': df_audit.nunique(),
        'Doublons (par colonne)': df_audit.apply(lambda x: x.duplicated().sum())
        
    })
    
    return stats
check_data_quality (df_audit)

Format du DataFrame (rows, cols): (1387077, 43)



,Type,Manquants,% Manquants,Uniques,Doublons (par colonne)
Identifiant de document,float64,1387077,100.00,0,1387076
Reference document,float64,1387077,100.00,0,1387076
1 Articles CGI,float64,1387077,100.00,0,1387076
2 Articles CGI,float64,1387077,100.00,0,1387076
3 Articles CGI,float64,1387077,100.00,0,1387076
4 Articles CGI,float64,1387077,100.00,0,1387076
5 Articles CGI,float64,1387077,100.00,0,1387076
No disposition,int64,0,0.00,43,1387034
Date mutation,str,0,0.00,173,1386904
Nature mutation,str,0,0.00,6,1387071


In [19]:
df_audit['Nature mutation'].value_counts()

Nature mutation
Vente                                 1310710
Vente en l'état futur d'achèvement      55448
Echange                                 16859
Vente terrain à bâtir                    2881
Adjudication                              928
Expropriation                             251
Name: count, dtype: int64

In [28]:
print(df_audit['Nature mutation'].value_counts())

Nature mutation
Vente                                 1310710
Vente en l'état futur d'achèvement      55448
Echange                                 16859
Vente terrain à bâtir                    2881
Adjudication                              928
Expropriation                             251
Name: count, dtype: int64


In [34]:
# Relevant filtering of real estate transactions
mutations_cibles = [
    'Vente', 
    "Vente en l'état futur d'achèvement", 
    'Vente terrain à bâtir'
]

# Filtring the DataFrame to keep only relevant real estate transactions
df_etude = df_audit[df_audit['Nature mutation'].isin(mutations_cibles)].copy()
# Summary of filtering
print(f"Volume initial : {len(df_audit)}")
print(f"Volume après filtrage : {len(df_etude)}")
print(f"Lignes supprimées : {len(df_audit) - len(df_etude)}")
# Aperçu des types de mutations restants
print("\nRépartition après filtrage :")
print(df_etude['Nature mutation'].value_counts())

Volume initial : 1387077
Volume après filtrage : 1369039
Lignes supprimées : 18038

Répartition après filtrage :
Nature mutation
Vente                                 1310710
Vente en l'état futur d'achèvement      55448
Vente terrain à bâtir                    2881
Name: count, dtype: int64


In [36]:
print(df_etude['Nature culture'].value_counts())

Nature culture
S     451122
T     134275
P      74130
J      45432
BT     43107
L      37187
AG     36121
AB     30466
VI     14778
BR     13827
VE     10664
BS      7299
PA      6971
B       4817
E       3626
BP      3435
BF      2184
PP      1012
PC       689
PH       571
BM       554
CH       392
CA       322
LB       106
PE        68
TP        64
BO        28
Name: count, dtype: int64


In [56]:
# 1. Définition des codes à haut potentiel
nature_cibles = ['S', 'L', 'J', 'BR']

# 2. Création du DataFrame filtré
df_immo = df_etude[df_etude['Nature culture'].isin(nature_cibles)].copy()

# 3. CORRECTION : On nettoie la colonne SANS écraser tout le DataFrame
df_immo['Nature culture'] = df_immo['Nature culture'].astype(str).str.strip()

# 4. Rapport de l'auditeur (Maintenant stats_culture fonctionnera !)
print("--- Analyse du Gisement Foncier Social ---")
stats_culture = df_immo['Nature culture'].value_counts()

for code, count in stats_culture.items():
    pct = (count / len(df_immo) * 100)
    print(f"Code {code}: {count} parcelles ({round(pct, 2)}%)")

print(f"\nTotal parcelles exploitables pour le projet : {len(df_immo)}")

--- Analyse du Gisement Foncier Social ---
Code S: 451122 parcelles (82.39%)
Code J: 45432 parcelles (8.3%)
Code L: 37187 parcelles (6.79%)
Code BR: 13827 parcelles (2.53%)

Total parcelles exploitables pour le projet : 547568


In [ ]:
check_data_quality (df_immo)


Format du DataFrame (rows, cols): (547568, 43)



,Type,Manquants,% Manquants,Uniques,Doublons (par colonne)
Identifiant de document,float64,547568,100.00,0,547567
Reference document,float64,547568,100.00,0,547567
1 Articles CGI,float64,547568,100.00,0,547567
2 Articles CGI,float64,547568,100.00,0,547567
3 Articles CGI,float64,547568,100.00,0,547567
4 Articles CGI,float64,547568,100.00,0,547567
5 Articles CGI,float64,547568,100.00,0,547567
No disposition,int64,0,0.00,13,547555
Date mutation,object,0,0.00,169,547399
Nature mutation,object,0,0.00,3,547565


In [64]:
print (df_immo.columns.tolist())

['Identifiant de document', 'Reference document', '1 Articles CGI', '2 Articles CGI', '3 Articles CGI', '4 Articles CGI', '5 Articles CGI', 'No disposition', 'Date mutation', 'Nature mutation', 'Valeur fonciere', 'No voie', 'B/T/Q', 'Type de voie', 'Code voie', 'Voie', 'Code postal', 'Commune', 'Code departement', 'Code commune', 'Prefixe de section', 'Section', 'No plan', 'No Volume', '1er lot', 'Surface Carrez du 1er lot', '2eme lot', 'Surface Carrez du 2eme lot', '3eme lot', 'Surface Carrez du 3eme lot', '4eme lot', 'Surface Carrez du 4eme lot', '5eme lot', 'Surface Carrez du 5eme lot', 'Nombre de lots', 'Code type local', 'Type local', 'Identifiant local', 'Surface reelle bati', 'Nombre pieces principales', 'Nature culture', 'Nature culture speciale', 'Surface terrain']


In [76]:
# 1. Liste des colonnes à supprimer (ton "bruit" administratif)
colonnes_a_supprimer = [
    'Identifiant de document', 'Reference document', 
    '1 Articles CGI', '2 Articles CGI', '3 Articles CGI', '4 Articles CGI', '5 Articles CGI', 
    'No Volume', 'No plan',
    '1er lot', 'Surface Carrez du 1er lot', 
    '2eme lot', 'Surface Carrez du 2eme lot', 
    '3eme lot', 'Surface Carrez du 3eme lot',
    '4eme lot', 'Surface Carrez du 4eme lot', 
    '5eme lot', 'Surface Carrez du 5eme lot', 'Nature culture speciale', 'Identifiant local', 'Prefixe de section', 'B/T/Q', 
    'No voie', 'Type de voie', 'Code voie', 'Voie', 'Code postal', 'Nombre pieces principales','Section','No disposition'
]

# 2. Suppression sécurisée
# errors='ignore' permet d'éviter un plantage si une colonne a déjà été supprimée
df_immo = df_immo.drop(columns=colonnes_a_supprimer, errors='ignore')

# 3. Rapport de l'auditeur
print(f"✅ Nettoyage effectué. Colonnes restantes : {len(df_immo.columns)}")
print(df_immo.columns.tolist())

✅ Nettoyage effectué. Colonnes restantes : 12
['Date mutation', 'Nature mutation', 'Valeur fonciere', 'Commune', 'Code departement', 'Code commune', 'Nombre de lots', 'Code type local', 'Type local', 'Surface reelle bati', 'Nature culture', 'Surface terrain']


In [77]:
df_immo.head()

,Date mutation,Nature mutation,Valeur fonciere,Commune,Code departement,Code commune,Nombre de lots,Code type local,Type local,Surface reelle bati,Nature culture,Surface terrain
0,07/01/2025,Vente,"468000,00",FARGES,01,158,0,NaN,NaN,NaN,J,78.0
1,07/01/2025,Vente,"468000,00",FARGES,01,158,0,1.0,Maison,111.0,S,133.0
2,07/01/2025,Vente,"468000,00",FARGES,01,158,0,3.0,Dépendance,0.0,S,133.0
3,06/01/2025,Vente,"180000,00",MONTANGES,01,257,0,NaN,NaN,NaN,S,46.0
4,06/01/2025,Vente,"180000,00",MONTANGES,01,257,0,NaN,NaN,NaN,J,17.0


In [78]:
check_data_quality (df_immo)

Format du DataFrame (rows, cols): (547568, 12)



,Type,Manquants,% Manquants,Uniques,Doublons (par colonne)
Date mutation,object,0,0.00,169,547399
Nature mutation,object,0,0.00,3,547565
Valeur fonciere,object,7447,1.36,35403,512164
Commune,object,0,0.00,26442,521126
Code departement,object,0,0.00,97,547471
Code commune,int64,0,0.00,894,546674
Nombre de lots,int64,0,0.00,2,547566
Code type local,float64,181370,33.12,4,547563
Type local,object,181370,33.12,4,547563
Surface reelle bati,float64,182010,33.24,2550,545017


In [79]:
print (df_immo.columns.tolist())

['Date mutation', 'Nature mutation', 'Valeur fonciere', 'Commune', 'Code departement', 'Code commune', 'Nombre de lots', 'Code type local', 'Type local', 'Surface reelle bati', 'Nature culture', 'Surface terrain']


## 02 exploration

In [2]:
import pandas as pd

# 1. Utilisation de nrows=100 pour une lecture instantanée
# 2. low_memory=False pour éviter les erreurs de type de colonnes
# 3. sep=',' ou '|' (vérifie ton fichier, souvent c'est la virgule ou la barre verticale)


In [29]:
df_ips_ = pd.read_csv('../data/raw/02_fr-en-ips-ecoles-ap2022.csv', sep=';')
df_income_ = pd.read_csv('../data/raw/02_FILO2021_DISP_COM.csv', sep=';', encoding='latin-1')

C:\Users\Utilisateur\AppData\Local\Temp\ipykernel_17288\1423481474.py:2: DtypeWarning: Columns (0: CODGEO) have mixed types. Specify dtype option on import or set low_memory=False.
  df_income_ = pd.read_csv('../data/raw/02_FILO2021_DISP_COM.csv', sep=';', encoding='latin-1')


In [30]:
check_data_quality (df_ips_)

Format du DataFrame (rows, cols): (97080, 23)



,Type,Manquants,% Manquants,Uniques,Doublons (par colonne)
num_ligne,float64,0,0.00,97080,0
Rentrée scolaire,str,0,0.00,3,97077
Code région,int64,0,0.00,18,97062
Région,str,0,0.00,18,97062
Code de l'académie,int64,0,0.00,30,97050
Académie,str,0,0.00,30,97050
Code du département,str,0,0.00,101,96979
Département,str,0,0.00,104,96976
Code INSEE de la commune,str,0,0.00,18324,78756
Nom de la commune,str,0,0.00,17766,79314


In [31]:
df_ips_

,num_ligne,Rentrée scolaire,Code région,Région,Code de l'académie,Académie,Code du département,Département,Code INSEE de la commune,Nom de la commune,...,IPS,IPS national privé,IPS national public,IPS national,IPS académique privé,IPS académique public,IPS académique,IPS départemental privé,IPS départemental public,IPS départemental
0,40004.0,2023-2024,16,OCCITANIE,16,TOULOUSE,31,HAUTE-GARONNE,31147,CLARAC,...,NS,120.3,102.9,105.5,119.8,108.4,109.9,133.6,114.3,116.2
1,40011.0,2023-2024,16,OCCITANIE,16,TOULOUSE,31,HAUTE-GARONNE,31157,CUGNAUX,...,115.6,120.3,102.9,105.5,119.8,108.4,109.9,133.6,114.3,116.2
2,40012.0,2023-2024,16,OCCITANIE,16,TOULOUSE,31,HAUTE-GARONNE,31159,LE CUING,...,97.8,120.3,102.9,105.5,119.8,108.4,109.9,133.6,114.3,116.2
3,40013.0,2023-2024,16,OCCITANIE,16,TOULOUSE,31,HAUTE-GARONNE,31160,DAUX,...,130.3,120.3,102.9,105.5,119.8,108.4,109.9,133.6,114.3,116.2
4,40017.0,2023-2024,16,OCCITANIE,16,TOULOUSE,31,HAUTE-GARONNE,31165,EAUNES,...,106.2,120.3,102.9,105.5,119.8,108.4,109.9,133.6,114.3,116.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97075,12485.0,2022-2023,17,PAYS DE LA LOIRE,17,NANTES,44,LOIRE-ATLANTIQUE,44143,REZE,...,120.5,119.3,102.9,105.4,112.0,104.5,107.2,118.1,110.6,113.2
97076,12486.0,2022-2023,17,PAYS DE LA LOIRE,17,NANTES,44,LOIRE-ATLANTIQUE,44143,REZE,...,122.2,119.3,102.9,105.4,112.0,104.5,107.2,118.1,110.6,113.2
97077,12488.0,2022-2023,17,PAYS DE LA LOIRE,17,NANTES,44,LOIRE-ATLANTIQUE,44144,RIAILLE,...,102.6,119.3,102.9,105.4,112.0,104.5,107.2,118.1,110.6,113.2
97078,12496.0,2022-2023,17,PAYS DE LA LOIRE,17,NANTES,44,LOIRE-ATLANTIQUE,44155,SAINT COLOMBAN,...,111.8,119.3,102.9,105.4,112.0,104.5,107.2,118.1,110.6,113.2


In [35]:
df_ips_.columns.tolist()

['num_ligne',
 'Rentrée scolaire',
 'Code région',
 'Région',
 "Code de l'académie",
 'Académie',
 'Code du département',
 'Département',
 'Code INSEE de la commune',
 'Nom de la commune',
 'UAI',
 "Nom de l'établissement",
 'Secteur',
 'IPS',
 'IPS national privé',
 'IPS national public',
 'IPS national',
 'IPS académique privé',
 'IPS académique public',
 'IPS académique',
 'IPS départemental privé',
 'IPS départemental public',
 'IPS départemental']

In [36]:
# 1 Define target columns to save memory
# target_columns based on business needs: Price, Location, Type, and Surface
target_ips_columns = [
    'Code INSEE de la commune', 'Secteur', 'IPS', 
    "Nom de l'établissement"
]

# Load the DVF file (using '|' separator as it's a .txt file from Government)
df_ips = pd.read_csv(
    '../data/raw/02_fr-en-ips-ecoles-ap2022.csv',
    # nrows=100, 
    usecols=target_ips_columns,
    sep=';', 
    #sep='|',
    low_memory=False
)



In [40]:
def clean_column_names(df):
    """
    Standardize DataFrame column names: 1. Convert to string / 2. Strip leading/trailing whitespaces
    3. Replace internal spaces with underscores / 4. Convert to lowercase
    """
    df.columns = (
        df.columns.astype(str)
                  .str.strip()
                  .str.replace(' ', '_', regex=False)
                  .str.lower()
    )
    return df

In [41]:
clean_column_names (df_ips)

,code_insee_de_la_commune,nom_de_l'établissement,secteur,ips
0,31147,ECOLE ELEMENTAIRE PUBLIQUE,public,NS
1,31157,ECOLE ELEMENTAIRE PUBLIQUE JEAN JAURES,public,115.6
2,31159,ECOLE ELEMENTAIRE PUBLIQUE,public,97.8
3,31160,ECOLE PRIMAIRE PUBLIQUE,public,130.3
4,31165,ECOLE ELEMENTAIRE PUBLIQUE JEAN DARGASSIES,public,106.2
...,...,...,...,...
97075,44143,ECOLE ELEMENTAIRE PUBLIQUE OUCHE DINIER,public,120.5
97076,44143,ECOLE ELEMENTAIRE PUBLIQUE ROGER SALENGRO,public,122.2
97077,44144,ECOLE PRIMAIRE PUBLIQUE ROBERT DOISNEAU,public,102.6
97078,44155,ECOLE PRIMAIRE PUBLIQUE JACQUES PREVERT,public,111.8


In [42]:
df_ips.to_csv('../data/clean/df_ips_processed.csv', index=False)

In [37]:
df_income_

,CODGEO,NBMEN21,NBPERS21,NBUC21,Q121,Q221,Q321,Q3_Q1,D121,D221,...,OPR6PTSA21,OPR6PCHO21,OPR6PBEN21,OPR6PPEN21,OPR6PPAT21,OPR6PPSOC21,OPR6PPFAM21,OPR6PPMINI21,OPR6PPLOGT21,OPR6PIMPOT21
0,1001,346,895,"590,8",s,25820,s,s,s,s,...,s,s,s,s,s,s,s,s,s,s
1,1002,115,266,"181,0",s,24480,s,s,s,s,...,s,s,s,s,s,s,s,s,s,s
2,1004,6855,15092,"10398,2",15800,21660,28430,12630,11890,14640,...,s,s,s,s,s,s,s,s,s,s
3,1005,800,2028,"1329,7",20010,24610,31180,11170,15560,18980,...,s,s,s,s,s,s,s,s,s,s
4,1006,51,107,"76,6",s,24210,s,s,s,s,...,s,s,s,s,s,s,s,s,s,s
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34924,97420,8593,23979,"15260,6",11970,17720,26400,14430,9060,11030,...,s,s,s,s,s,s,s,s,s,s
34925,97421,2472,6847,"4379,7",9860,13280,19240,9380,6810,9150,...,s,s,s,s,s,s,s,s,s,s
34926,97422,31239,79757,"52192,7",11440,16560,25020,13580,8660,10670,...,s,s,s,s,s,s,s,s,s,s
34927,97423,2505,7019,"4507,7",11480,16680,24240,12750,8570,10760,...,s,s,s,s,s,s,s,s,s,s


In [38]:
check_data_quality(df_income_)

Format du DataFrame (rows, cols): (34929, 732)



,Type,Manquants,% Manquants,Uniques,Doublons (par colonne)
CODGEO,object,0,0.0,34929,0
NBMEN21,str,0,0.0,3814,31115
NBPERS21,str,0,0.0,5749,29180
NBUC21,str,0,0.0,13761,21168
Q121,str,0,0.0,1232,33697
...,...,...,...,...,...
OPR6PPSOC21,str,0,0.0,3,34926
OPR6PPFAM21,str,0,0.0,3,34926
OPR6PPMINI21,str,0,0.0,3,34926
OPR6PPLOGT21,str,0,0.0,3,34926


In [39]:
df_income_.columns.tolist()

['CODGEO',
 'NBMEN21',
 'NBPERS21',
 'NBUC21',
 'Q121',
 'Q221',
 'Q321',
 'Q3_Q1',
 'D121',
 'D221',
 'D321',
 'D421',
 'D621',
 'D721',
 'D821',
 'D921',
 'RD',
 'S80S2021',
 'GI21',
 'PACT21',
 'PTSA21',
 'PCHO21',
 'PBEN21',
 'PPEN21',
 'PPAT21',
 'PPSOC21',
 'PPFAM21',
 'PPMINI21',
 'PPLOGT21',
 'PIMPOT21',
 'AGE1Q121',
 'AGE1Q221',
 'AGE1Q321',
 'AGE1Q3_Q1',
 'AGE1D121',
 'AGE1D221',
 'AGE1D321',
 'AGE1D421',
 'AGE1D621',
 'AGE1D721',
 'AGE1D821',
 'AGE1D921',
 'AGE1RD',
 'AGE1S80S2021',
 'AGE1GI21',
 'AGE1PACT21',
 'AGE1PTSA21',
 'AGE1PCHO21',
 'AGE1PBEN21',
 'AGE1PPEN21',
 'AGE1PPAT21',
 'AGE1PPSOC21',
 'AGE1PPFAM21',
 'AGE1PPMINI21',
 'AGE1PPLOGT21',
 'AGE1PIMPOT21',
 'AGE2Q121',
 'AGE2Q221',
 'AGE2Q321',
 'AGE2Q3_Q1',
 'AGE2D121',
 'AGE2D221',
 'AGE2D321',
 'AGE2D421',
 'AGE2D621',
 'AGE2D721',
 'AGE2D821',
 'AGE2D921',
 'AGE2RD',
 'AGE2S80S2021',
 'AGE2GI21',
 'AGE2PACT21',
 'AGE2PTSA21',
 'AGE2PCHO21',
 'AGE2PBEN21',
 'AGE2PPEN21',
 'AGE2PPAT21',
 'AGE2PPSOC21',
 'AGE2PPFAM

In [43]:
# 1 Define target columns to save memory
# target_columns based on business needs: Price, Location, Type, and Surface
target_income_columns = [
    'CODGEO', 'NBMEN21', 'NBPERS21', 
    "NBUC21", 'Q221', 'PIMPOT21', 'TYM1Q221', 'TYM3Q221'
]

# Load the DVF file (using '|' separator as it's a .txt file from Government)
df_income = pd.read_csv(
    '../data/raw/02_FILO2021_DISP_COM.csv',
    # nrows=100, 
    usecols=target_income_columns,
    sep=';', 
    #sep='|',
    low_memory=False
)

In [46]:
df_income = df_income.rename(columns={
    'CODGEO': 'insee_code',
    'NBMEN21': 'nb_households',            # Nombre de foyers fiscaux (Ménages)
    'NBPERS21': 'nb_inhabitants',          # Le nombre total d'habitant
    'NBUC21': 'consumption_units',         # La pondération adulte/enfant (Standard INSEE)
    'Q221': 'median_income',               # Le revenu médian global de la zone
    'PIMPOT21': 'taxable_households_rate', # % de ménages payant l'impôt (Solvabilité)
    'TYM1Q221': 'solo_median_income',      # Revenu médian des célibataires
    'TYM3Q221': 'family_median_income'     # Revenu médian des couples avec enfants
})

In [47]:
df_income.to_csv('../data/clean/df_income_processed.csv', index=False)

In [ ]:
df_income